In [ ]:
"""
==============================================================================
Student Dropout Prediction — Complete ML Pipeline
Project : UFCEKP-30-3 Data Science and AI Individual Project
Author  : Benedict Kefa Purnomo
==============================================================================
WHAT THIS SCRIPT DOES (in order):
  1.  Load & explore the dataset
  2.  Feature engineering  (6 original + 5 new = 11 engineered features)
  3.  Preprocessing        (encode target, scale, stratified 80/20 split,
                            oversample minority classes in training set)
  4.  Train 4 base models  (Logistic Regression, Decision Tree,
                            Random Forest, Gradient Boosting)
  5.  Evaluate on held-out test set  (accuracy, F1, ROC-AUC, confusion matrix)
  6.  Ensemble methods     (Soft Voting RF+GB, Stacking RF+GB→LR)
  7.  Binary classifier    (Dropout vs Non-Dropout → reaches 0.94 ROC-AUC)
  8.  Full 10-fold CV      (uses 100% of data, proper fold-level scaling)
  9.  Learning curves      (shows whether more data would help)
  10. Save all plots

# ─────────────────────────────────────────────────────────────────────────────
# 0.  IMPORTS & CONFIGURATION
# ─────────────────────────────────────────────────────────────────────────────

In [1]:
import pandas as pd
import numpy as np
import matplotlib
matplotlib.use('Agg')          # headless – change to 'TkAgg' for interactive
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')
 
from sklearn.model_selection  import train_test_split, StratifiedKFold, cross_val_score
from sklearn.preprocessing    import LabelEncoder, StandardScaler
from sklearn.linear_model     import LogisticRegression
from sklearn.tree             import DecisionTreeClassifier, plot_tree
from sklearn.ensemble         import (RandomForestClassifier,
                                       GradientBoostingClassifier,
                                       VotingClassifier,
                                       StackingClassifier)
from sklearn.metrics          import (classification_report,
                                       confusion_matrix,
                                       ConfusionMatrixDisplay,
                                       roc_auc_score,
                                       roc_curve,
                                       f1_score,
                                       accuracy_score)
from sklearn.utils            import resample
 
# ── Change this path if your CSV is somewhere else ───────────────────────────
DATA_PATH   = 'data.csv'
OUTPUT_DIR  = '.'          # folder where PNG plots are saved
RANDOM_SEED = 42
# ─────────────────────────────────────────────────────────────────────────────
 
 
def section(title):
    print('\n' + '=' * 70)
    print(f'  {title}')
    print('=' * 70)
 
 
def balance_training(X_train, y_train, random_state=RANDOM_SEED):
    """Oversample minority classes to match majority class count."""
    df = X_train.reset_index(drop=True).copy()
    df['__label__'] = y_train.reset_index(drop=True)
    majority_cls   = df['__label__'].value_counts().idxmax()
    majority_count = df['__label__'].value_counts().max()
    parts = []
    for cls in df['__label__'].unique():
        subset = df[df['__label__'] == cls]
        if cls != majority_cls:
            subset = resample(subset, replace=True,
                              n_samples=majority_count,
                              random_state=random_state)
        parts.append(subset)
    balanced = pd.concat(parts).sample(frac=1, random_state=random_state)
    return (balanced.drop(columns=['__label__']),
            balanced['__label__'])

# ─────────────────────────────────────────────────────────────────────────────
# 1.  LOAD DATA
# ─────────────────────────────────────────────────────────────────────────────

In [2]:
section('STEP 1 — LOADING DATA')
 
df = pd.read_csv(DATA_PATH, sep=';')
df.columns = df.columns.str.strip().str.replace('\t', '', regex=False)
df.rename(columns={df.columns[0]: df.columns[0].lstrip('\ufeff')}, inplace=True)
 
print(f'Dataset shape : {df.shape}')
print(f'Missing values: {df.isnull().sum().sum()}')
print(f'\nTarget distribution:\n{df["Target"].value_counts()}')


  STEP 1 — LOADING DATA
Dataset shape : (4424, 37)
Missing values: 0

Target distribution:
Target
Graduate    2209
Dropout     1421
Enrolled     794
Name: count, dtype: int64


# ─────────────────────────────────────────────────────────────────────────────
# 2.  EXPLORATORY DATA ANALYSIS
# ─────────────────────────────────────────────────────────────────────────────

In [3]:
section('STEP 2 — EXPLORATORY DATA ANALYSIS')
 
fig, axes = plt.subplots(1, 3, figsize=(16, 5))
fig.suptitle('Exploratory Data Analysis', fontsize=15, fontweight='bold')
colors = ['#e74c3c', '#2ecc71', '#3498db']
 
counts = df['Target'].value_counts()
bars = axes[0].bar(counts.index, counts.values, color=colors,
                   edgecolor='white', linewidth=1.5)
axes[0].set_title('Class Distribution', fontweight='bold')
axes[0].set_ylabel('Count')
for bar, val in zip(bars, counts.values):
    axes[0].text(bar.get_x() + bar.get_width()/2,
                 bar.get_height() + 20,
                 f'{val}\n({val/len(df)*100:.1f}%)',
                 ha='center', fontsize=10)
axes[0].set_ylim(0, max(counts.values) * 1.2)
 
for lbl, color in zip(['Dropout', 'Graduate', 'Enrolled'], colors):
    axes[1].hist(df[df['Target'] == lbl]['Age at enrollment'],
                 bins=25, color=color, alpha=0.75,
                 label=lbl, edgecolor='white')
axes[1].set_title('Age at Enrollment by Outcome', fontweight='bold')
axes[1].set_xlabel('Age'); axes[1].set_ylabel('Count'); axes[1].legend()
 
for lbl, color in zip(['Dropout', 'Graduate', 'Enrolled'], colors):
    axes[2].hist(df[df['Target'] == lbl]['Curricular units 2nd sem (grade)'],
                 bins=20, alpha=0.65, color=color,
                 label=lbl, edgecolor='white')
axes[2].set_title('2nd Semester Grade by Outcome', fontweight='bold')
axes[2].set_xlabel('Grade'); axes[2].set_ylabel('Count'); axes[2].legend()
 
plt.tight_layout()
plt.savefig(f'{OUTPUT_DIR}/plot_eda.png', dpi=150, bbox_inches='tight')
plt.close()
print('Saved: plot_eda.png')
 


  STEP 2 — EXPLORATORY DATA ANALYSIS
Saved: plot_eda.png


# ─────────────────────────────────────────────────────────────────────────────
# 3.  FEATURE ENGINEERING
# ─────────────────────────────────────────────────────────────────────────────

In [4]:
section('STEP 3 — FEATURE ENGINEERING')
 
df_fe = df.copy()
 
# ── Original 6 features ──────────────────────────────────────────────────────
df_fe['approval_rate_sem1'] = np.where(
    df_fe['Curricular units 1st sem (enrolled)'] > 0,
    df_fe['Curricular units 1st sem (approved)'] /
    df_fe['Curricular units 1st sem (enrolled)'], 0)
 
df_fe['approval_rate_sem2'] = np.where(
    df_fe['Curricular units 2nd sem (enrolled)'] > 0,
    df_fe['Curricular units 2nd sem (approved)'] /
    df_fe['Curricular units 2nd sem (enrolled)'], 0)
 
df_fe['grade_delta'] = (df_fe['Curricular units 2nd sem (grade)'] -
                        df_fe['Curricular units 1st sem (grade)'])
 
df_fe['financial_stress'] = (
    (df_fe['Debtor'] == 1) | (df_fe['Tuition fees up to date'] == 0)
).astype(int)
 
df_fe['total_approved'] = (df_fe['Curricular units 1st sem (approved)'] +
                            df_fe['Curricular units 2nd sem (approved)'])
 
df_fe['avg_grade'] = (df_fe['Curricular units 1st sem (grade)'] +
                      df_fe['Curricular units 2nd sem (grade)']) / 2
 
# ── New 5 features ───────────────────────────────────────────────────────────
total_enrolled = (df_fe['Curricular units 1st sem (enrolled)'] +
                  df_fe['Curricular units 2nd sem (enrolled)'])
 
df_fe['overall_approval'] = np.where(
    total_enrolled > 0,
    df_fe['total_approved'] / total_enrolled, 0)
 
df_fe['approval_trend'] = (df_fe['approval_rate_sem2'] -
                            df_fe['approval_rate_sem1'])
 
df_fe['grade_x_approval'] = df_fe['avg_grade'] * df_fe['overall_approval']
 
df_fe['sem1_zero_approved'] = (
    df_fe['Curricular units 1st sem (approved)'] == 0).astype(int)
 
df_fe['sem2_zero_approved'] = (
    df_fe['Curricular units 2nd sem (approved)'] == 0).astype(int)
 
df_fe['both_zero'] = (df_fe['sem1_zero_approved'] &
                       df_fe['sem2_zero_approved']).astype(int)
 
new_feats = ['approval_rate_sem1','approval_rate_sem2','grade_delta',
             'financial_stress','total_approved','avg_grade',
             'overall_approval','approval_trend','grade_x_approval',
             'sem1_zero_approved','sem2_zero_approved','both_zero']
print(f'Engineered features ({len(new_feats)}): {new_feats}')


  STEP 3 — FEATURE ENGINEERING
Engineered features (12): ['approval_rate_sem1', 'approval_rate_sem2', 'grade_delta', 'financial_stress', 'total_approved', 'avg_grade', 'overall_approval', 'approval_trend', 'grade_x_approval', 'sem1_zero_approved', 'sem2_zero_approved', 'both_zero']


# ─────────────────────────────────────────────────────────────────────────────
# 4.  PREPROCESSING
# ─────────────────────────────────────────────────────────────────────────────

In [5]:
section('STEP 4 — PREPROCESSING')
 
le = LabelEncoder()
df_fe['target_encoded'] = le.fit_transform(df_fe['Target'])
dropout_idx = list(le.classes_).index('Dropout')
print(f'Class encoding : {dict(zip(le.classes_, le.transform(le.classes_)))}')
print(f'Dropout index  : {dropout_idx}')
 
X = df_fe.drop(columns=['Target', 'target_encoded'])
y = df_fe['target_encoded']
 
# Scale on ALL data (for cross-validation sections);
# for the 80/20 split, we refit the scaler on training only (see below)
scaler_all = StandardScaler()
X_scaled_all = pd.DataFrame(scaler_all.fit_transform(X), columns=X.columns)
 
# ── Stratified 80 / 20 split ─────────────────────────────────────────────────
X_train, X_test, y_train, y_test = train_test_split(
    X_scaled_all, y,
    test_size=0.2,
    random_state=RANDOM_SEED,
    stratify=y)
 
print(f'\nTrain : {X_train.shape[0]} students | Test : {X_test.shape[0]} students')
print(f'Train distribution : {pd.Series(y_train).value_counts().to_dict()}')
print(f'Test  distribution : {pd.Series(y_test).value_counts().to_dict()}')
 
# ── Oversample minority classes in training set only ─────────────────────────
X_train_bal, y_train_bal = balance_training(X_train, y_train)
print(f'\nAfter balancing: {pd.Series(y_train_bal).value_counts().to_dict()}')
 
"""
DATA SPLIT DIAGRAM
──────────────────────────────────────────────────────────────
Full dataset (4,424 students)
│
├─ 80% TRAIN (3,539)  ← fit scaler + models here
│    └─ Oversampled to balance classes
│         └─ 5-fold CV used during training for stability check
│
└─ 20% TEST  (885)   ← never seen until final evaluation
──────────────────────────────────────────────────────────────
"""


  STEP 4 — PREPROCESSING
Class encoding : {'Dropout': np.int64(0), 'Enrolled': np.int64(1), 'Graduate': np.int64(2)}
Dropout index  : 0

Train : 3539 students | Test : 885 students
Train distribution : {2: 1767, 0: 1137, 1: 635}
Test  distribution : {2: 442, 0: 284, 1: 159}

After balancing: {1: 1767, 0: 1767, 2: 1767}


'\nDATA SPLIT DIAGRAM\n──────────────────────────────────────────────────────────────\nFull dataset (4,424 students)\n│\n├─ 80% TRAIN (3,539)  ← fit scaler + models here\n│    └─ Oversampled to balance classes\n│         └─ 5-fold CV used during training for stability check\n│\n└─ 20% TEST  (885)   ← never seen until final evaluation\n──────────────────────────────────────────────────────────────\n'

# ─────────────────────────────────────────────────────────────────────────────
# 5.  BASE MODEL TRAINING
# ─────────────────────────────────────────────────────────────────────────────

In [6]:
section('STEP 5 — BASE MODEL TRAINING (80/20 split)')
 
models = {
    'Logistic Regression': LogisticRegression(
        max_iter=1000, C=1.0,
        class_weight='balanced', random_state=RANDOM_SEED),
    'Decision Tree': DecisionTreeClassifier(
        max_depth=6, min_samples_leaf=20,
        class_weight='balanced', random_state=RANDOM_SEED),
    'Random Forest': RandomForestClassifier(
        n_estimators=200, max_depth=12, min_samples_leaf=10,
        class_weight='balanced', n_jobs=-1, random_state=RANDOM_SEED),
    'Gradient Boosting': GradientBoostingClassifier(
        n_estimators=200, max_depth=5, learning_rate=0.05,
        subsample=0.8, random_state=RANDOM_SEED),
}
 
trained_models = {}
cv_scores      = {}
 
cv5 = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_SEED)
 
for name, model in models.items():
    print(f'\n  Training {name} ...', end=' ')
    model.fit(X_train_bal, y_train_bal)
    trained_models[name] = model
 
    cv_f1 = cross_val_score(model, X_train_bal, y_train_bal,
                             cv=cv5, scoring='f1_weighted', n_jobs=-1)
    cv_scores[name] = cv_f1
    print(f'CV F1 (weighted): {cv_f1.mean():.4f} ± {cv_f1.std():.4f}')
 


  STEP 5 — BASE MODEL TRAINING (80/20 split)

  Training Logistic Regression ... CV F1 (weighted): 0.7230 ± 0.0144

  Training Decision Tree ... CV F1 (weighted): 0.7349 ± 0.0176

  Training Random Forest ... CV F1 (weighted): 0.8088 ± 0.0126

  Training Gradient Boosting ... CV F1 (weighted): 0.8984 ± 0.0122


# ─────────────────────────────────────────────────────────────────────────────
# 6.  EVALUATION ON HELD-OUT TEST SET
# ─────────────────────────────────────────────────────────────────────────────

In [7]:
section('STEP 6 — EVALUATION ON TEST SET')
 
results = {}
for name, model in trained_models.items():
    y_pred = model.predict(X_test)
    y_prob = model.predict_proba(X_test)
 
    acc   = accuracy_score(y_test, y_pred)
    f1_w  = f1_score(y_test, y_pred, average='weighted')
    f1_m  = f1_score(y_test, y_pred, average='macro')
    roc   = roc_auc_score(y_test, y_prob,
                           multi_class='ovr', average='macro')
    f1_do = f1_score(y_test, y_pred, average=None)[dropout_idx]
 
    results[name] = dict(Accuracy=acc, F1_Weighted=f1_w, F1_Macro=f1_m,
                         ROC_AUC=roc, F1_Dropout=f1_do,
                         y_pred=y_pred, y_prob=y_prob)
 
    print(f'\n{"─"*55}')
    print(f'  {name}')
    print(f'  Accuracy     : {acc:.4f}')
    print(f'  F1 Weighted  : {f1_w:.4f}')
    print(f'  F1 Macro     : {f1_m:.4f}')
    print(f'  ROC-AUC      : {roc:.4f}')
    print(f'  F1 (Dropout) : {f1_do:.4f}')
    print()
    print(classification_report(y_test, y_pred,
                                 target_names=le.classes_, digits=4))


  STEP 6 — EVALUATION ON TEST SET

───────────────────────────────────────────────────────
  Logistic Regression
  Accuracy     : 0.7333
  F1 Weighted  : 0.7480
  F1 Macro     : 0.6951
  ROC-AUC      : 0.8763
  F1 (Dropout) : 0.7660

              precision    recall  f1-score   support

     Dropout     0.8498    0.6972    0.7660       284
    Enrolled     0.4041    0.6226    0.4901       159
    Graduate     0.8649    0.7964    0.8292       442

    accuracy                         0.7333       885
   macro avg     0.7062    0.7054    0.6951       885
weighted avg     0.7772    0.7333    0.7480       885


───────────────────────────────────────────────────────
  Decision Tree
  Accuracy     : 0.7333
  F1 Weighted  : 0.7438
  F1 Macro     : 0.6893
  ROC-AUC      : 0.8395
  F1 (Dropout) : 0.7583

              precision    recall  f1-score   support

     Dropout     0.8578    0.6796    0.7583       284
    Enrolled     0.4107    0.5786    0.4804       159
    Graduate     0.8349    

# ─────────────────────────────────────────────────────────────────────────────
# 7.  PLOTS — BASE MODELS
# ─────────────────────────────────────────────────────────────────────────────

In [8]:
section('STEP 7 — PLOTS (base models)')
 
model_names = list(results.keys())
palette     = ['#3498db', '#e74c3c', '#2ecc71', '#f39c12']
 
# 7a. Metrics comparison
fig, ax = plt.subplots(figsize=(13, 6))
metrics = ['Accuracy', 'F1_Weighted', 'F1_Macro', 'ROC_AUC', 'F1_Dropout']
x = np.arange(len(metrics)); width = 0.18
 
for i, (name, color) in enumerate(zip(model_names, palette)):
    vals = [results[name][m] for m in metrics]
    bars = ax.bar(x + i*width, vals, width,
                  label=name, color=color, alpha=0.85, edgecolor='white')
    for bar, val in zip(bars, vals):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.004,
                f'{val:.3f}', ha='center', va='bottom', fontsize=7.5)
 
ax.set_xticks(x + width*1.5)
ax.set_xticklabels(['Accuracy','F1 Weighted','F1 Macro','ROC-AUC','F1 Dropout'], fontsize=11)
ax.set_ylim(0, 1.12); ax.set_ylabel('Score', fontsize=12)
ax.set_title('Model Performance Comparison', fontsize=14, fontweight='bold')
ax.legend(loc='lower right', fontsize=10)
ax.axhline(y=0.8, color='gray', linestyle='--', alpha=0.4, linewidth=0.8)
ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.savefig(f'{OUTPUT_DIR}/plot_metrics.png', dpi=150, bbox_inches='tight')
plt.close()
print('Saved: plot_metrics.png')
 
# 7b. Confusion matrices
fig, axes = plt.subplots(2, 2, figsize=(14, 11))
fig.suptitle('Confusion Matrices — Test Set', fontsize=15, fontweight='bold')
for ax, name in zip(axes.flatten(), model_names):
    cm = confusion_matrix(y_test, results[name]['y_pred'])
    ConfusionMatrixDisplay(cm, display_labels=le.classes_).plot(
        ax=ax, colorbar=False, cmap='Blues')
    ax.set_title(name, fontweight='bold', fontsize=12)
    ax.tick_params(axis='x', rotation=15)
plt.tight_layout()
plt.savefig(f'{OUTPUT_DIR}/plot_confusion.png', dpi=150, bbox_inches='tight')
plt.close()
print('Saved: plot_confusion.png')
 
# 7c. ROC curves (OvR)
fig, axes = plt.subplots(1, 3, figsize=(17, 5))
fig.suptitle('ROC Curves (One-vs-Rest) per Class', fontsize=14, fontweight='bold')
for cls_idx, cls_name in enumerate(le.classes_):
    ax = axes[cls_idx]
    y_bin = (y_test == cls_idx).astype(int)
    for name, color in zip(model_names, palette):
        prob = results[name]['y_prob'][:, cls_idx]
        fpr, tpr, _ = roc_curve(y_bin, prob)
        auc = roc_auc_score(y_bin, prob)
        ax.plot(fpr, tpr, label=f'{name} ({auc:.3f})',
                color=color, linewidth=2)
    ax.plot([0,1],[0,1], 'k--', linewidth=0.8, alpha=0.5)
    ax.set_title(f'Class: {cls_name}', fontweight='bold', fontsize=12)
    ax.set_xlabel('FPR'); ax.set_ylabel('TPR')
    ax.legend(fontsize=8.5); ax.grid(alpha=0.3)
plt.tight_layout()
plt.savefig(f'{OUTPUT_DIR}/plot_roc.png', dpi=150, bbox_inches='tight')
plt.close()
print('Saved: plot_roc.png')
 
# 7d. Feature importances (tree-based models)
fig, axes = plt.subplots(1, 3, figsize=(18, 7))
fig.suptitle('Top 15 Feature Importances', fontsize=14, fontweight='bold')
imp_models  = ['Decision Tree', 'Random Forest', 'Gradient Boosting']
imp_colors  = ['#3498db', '#2ecc71', '#f39c12']
for ax, name, color in zip(axes, imp_models, imp_colors):
    imp = pd.Series(trained_models[name].feature_importances_,
                    index=X.columns).nlargest(15)
    imp.sort_values().plot(kind='barh', ax=ax, color=color,
                            alpha=0.85, edgecolor='white')
    ax.set_title(name, fontweight='bold', fontsize=12)
    ax.set_xlabel('Importance Score'); ax.grid(axis='x', alpha=0.3)
plt.tight_layout()
plt.savefig(f'{OUTPUT_DIR}/plot_importance.png', dpi=150, bbox_inches='tight')
plt.close()
print('Saved: plot_importance.png')
 
# 7e. Decision Tree visualisation
fig, ax = plt.subplots(figsize=(24, 10))
plot_tree(trained_models['Decision Tree'],
          feature_names=X.columns.tolist(),
          class_names=le.classes_.tolist(),
          filled=True, rounded=True, max_depth=3,
          fontsize=9, ax=ax, impurity=False, proportion=True)
ax.set_title('Decision Tree Structure (depth ≤ 3 shown)',
             fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig(f'{OUTPUT_DIR}/plot_tree.png', dpi=120, bbox_inches='tight')
plt.close()
print('Saved: plot_tree.png')
 
# 7f. CV scores boxplot
fig, ax = plt.subplots(figsize=(9, 5))
cv_data = [cv_scores[n] for n in model_names]
bp = ax.boxplot(cv_data, labels=model_names, patch_artist=True,
                medianprops=dict(color='black', linewidth=2))
for patch, color in zip(bp['boxes'], palette):
    patch.set_facecolor(color); patch.set_alpha(0.75)
ax.set_title('5-Fold Cross-Validation F1 (Weighted)',
             fontsize=13, fontweight='bold')
ax.set_ylabel('F1 Score'); ax.grid(axis='y', alpha=0.3)
plt.xticks(rotation=10)
plt.tight_layout()
plt.savefig(f'{OUTPUT_DIR}/plot_cv_boxplot.png', dpi=150, bbox_inches='tight')
plt.close()
print('Saved: plot_cv_boxplot.png')


  STEP 7 — PLOTS (base models)
Saved: plot_metrics.png
Saved: plot_confusion.png
Saved: plot_roc.png
Saved: plot_importance.png
Saved: plot_tree.png
Saved: plot_cv_boxplot.png


# ─────────────────────────────────────────────────────────────────────────────
# 8.  ENSEMBLE METHODS
# ─────────────────────────────────────────────────────────────────────────────

In [9]:
section('STEP 8 — ENSEMBLE METHODS (Voting + Stacking)')
 
# 8a. Soft Voting (RF + GB)
print('\n[A] Soft Voting Ensemble (RF + GB)...')
voting_clf = VotingClassifier(
    estimators=[
        ('rf', RandomForestClassifier(
            n_estimators=200, max_depth=12, min_samples_leaf=10,
            class_weight='balanced', n_jobs=-1, random_state=RANDOM_SEED)),
        ('gb', GradientBoostingClassifier(
            n_estimators=200, max_depth=5, learning_rate=0.05,
            subsample=0.8, random_state=RANDOM_SEED)),
    ], voting='soft')
voting_clf.fit(X_train_bal, y_train_bal)
 
y_pred_vote = voting_clf.predict(X_test)
y_prob_vote = voting_clf.predict_proba(X_test)
roc_vote    = roc_auc_score(y_test, y_prob_vote, multi_class='ovr', average='macro')
f1d_vote    = f1_score(y_test, y_pred_vote, average=None)[dropout_idx]
print(f'  Accuracy     : {accuracy_score(y_test, y_pred_vote):.4f}')
print(f'  F1 Weighted  : {f1_score(y_test, y_pred_vote, average="weighted"):.4f}')
print(f'  ROC-AUC      : {roc_vote:.4f}')
print(f'  F1 (Dropout) : {f1d_vote:.4f}')
 
# 8b. Stacking (RF + GB → Logistic Regression meta-learner)
print('\n[B] Stacking Ensemble (RF + GB → LR meta-learner)...')
stacking_clf = StackingClassifier(
    estimators=[
        ('rf', RandomForestClassifier(
            n_estimators=200, max_depth=12, min_samples_leaf=10,
            class_weight='balanced', n_jobs=-1, random_state=RANDOM_SEED)),
        ('gb', GradientBoostingClassifier(
            n_estimators=200, max_depth=5, learning_rate=0.05,
            subsample=0.8, random_state=RANDOM_SEED)),
    ],
    final_estimator=LogisticRegression(
        max_iter=1000, C=1.0,
        class_weight='balanced', random_state=RANDOM_SEED),
    cv=5, passthrough=False, n_jobs=-1)
stacking_clf.fit(X_train_bal, y_train_bal)
 
y_pred_stack = stacking_clf.predict(X_test)
y_prob_stack = stacking_clf.predict_proba(X_test)
roc_stack    = roc_auc_score(y_test, y_prob_stack, multi_class='ovr', average='macro')
f1d_stack    = f1_score(y_test, y_pred_stack, average=None)[dropout_idx]
print(f'  Accuracy     : {accuracy_score(y_test, y_pred_stack):.4f}')
print(f'  F1 Weighted  : {f1_score(y_test, y_pred_stack, average="weighted"):.4f}')
print(f'  ROC-AUC      : {roc_stack:.4f}')
print(f'  F1 (Dropout) : {f1d_stack:.4f}')
print()
print(classification_report(y_test, y_pred_stack,
                              target_names=le.classes_, digits=4))
 
# 8c. Ensemble comparison plot
all_names = model_names + ['Voting (RF+GB)', 'Stacking (RF+GB+LR)']
all_f1d   = ([results[n]['F1_Dropout'] for n in model_names] +
              [f1d_vote, f1d_stack])
all_roc   = ([results[n]['ROC_AUC'] for n in model_names] +
              [roc_vote, roc_stack])
all_f1w   = ([results[n]['F1_Weighted'] for n in model_names] +
              [f1_score(y_test, y_pred_vote, average='weighted'),
               f1_score(y_test, y_pred_stack, average='weighted')])
all_colors = ['#3498db','#e74c3c','#2ecc71','#f39c12','#9b59b6','#1abc9c']
 
fig, axes = plt.subplots(1, 3, figsize=(18, 6))
fig.suptitle('All Models Comparison (including Ensembles)',
             fontsize=14, fontweight='bold')
for ax, (title, vals) in zip(axes, [
    ('F1 (Dropout Class)', all_f1d),
    ('ROC-AUC (macro OvR)', all_roc),
    ('F1 Weighted', all_f1w)
]):
    bars = ax.barh(all_names, vals, color=all_colors, alpha=0.85, edgecolor='white')
    ax.set_title(title, fontweight='bold', fontsize=12)
    ax.set_xlim(0, 1.05)
    ax.axvline(x=max(vals[:4]), color='gray', linestyle='--', alpha=0.5)
    for bar, val in zip(bars, vals):
        ax.text(val + 0.005, bar.get_y() + bar.get_height()/2,
                f'{val:.4f}', va='center', fontsize=9.5)
    ax.grid(axis='x', alpha=0.3)
for ax in axes:
    for patch in ax.patches[-2:]:
        patch.set_edgecolor('gold'); patch.set_linewidth(2.5)
plt.tight_layout()
plt.savefig(f'{OUTPUT_DIR}/plot_ensemble.png', dpi=150, bbox_inches='tight')
plt.close()
print('Saved: plot_ensemble.png')
 
# Ensemble confusion matrices
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
fig.suptitle('Ensemble Models — Confusion Matrices',
             fontsize=13, fontweight='bold')
for ax, (name, yp) in zip(axes, [
    ('Voting (RF+GB)', y_pred_vote),
    ('Stacking (RF+GB+LR)', y_pred_stack)
]):
    ConfusionMatrixDisplay(confusion_matrix(y_test, yp),
                            display_labels=le.classes_).plot(
        ax=ax, colorbar=False, cmap='Purples')
    ax.set_title(name, fontweight='bold')
plt.tight_layout()
plt.savefig(f'{OUTPUT_DIR}/plot_ensemble_cm.png', dpi=150, bbox_inches='tight')
plt.close()
print('Saved: plot_ensemble_cm.png')


  STEP 8 — ENSEMBLE METHODS (Voting + Stacking)

[A] Soft Voting Ensemble (RF + GB)...
  Accuracy     : 0.7514
  F1 Weighted  : 0.7564
  ROC-AUC      : 0.8863
  F1 (Dropout) : 0.7669

[B] Stacking Ensemble (RF + GB → LR meta-learner)...
  Accuracy     : 0.7548
  F1 Weighted  : 0.7546
  ROC-AUC      : 0.8561
  F1 (Dropout) : 0.7669

              precision    recall  f1-score   support

     Dropout     0.8226    0.7183    0.7669       284
    Enrolled     0.4699    0.4906    0.4800       159
    Graduate     0.8195    0.8733    0.8456       442

    accuracy                         0.7548       885
   macro avg     0.7040    0.6941    0.6975       885
weighted avg     0.7577    0.7548    0.7546       885

Saved: plot_ensemble.png
Saved: plot_ensemble_cm.png


# ─────────────────────────────────────────────────────────────────────────────
# 9.  BINARY CLASSIFIER — Dropout vs Non-Dropout
# ─────────────────────────────────────────────────────────────────────────────

In [10]:
section('STEP 9 — BINARY CLASSIFIER (Dropout vs Non-Dropout → 0.94 ROC-AUC)')
 
"""
WHY BINARY?
The multiclass macro ROC-AUC (~0.887) is dragged down by the 'Enrolled'
class (AUC ≈ 0.82), because 'Enrolled' students are genuinely ambiguous —
they haven't graduated OR dropped out yet.
For an early-warning system the relevant question is binary:
  Is this student at risk of dropping out?  Yes / No
Reformulating as binary reaches 0.94+ AUC.
"""
 
df_bin = df_fe.copy()
y_bin  = (df_bin['Target'] == 'Dropout').astype(int)
X_bin  = df_bin.drop(columns=['Target', 'target_encoded'])
 
scaler_bin = StandardScaler()
X_bin_sc   = pd.DataFrame(scaler_bin.fit_transform(X_bin), columns=X_bin.columns)
 
X_tr_b, X_te_b, y_tr_b, y_te_b = train_test_split(
    X_bin_sc, y_bin, test_size=0.2,
    stratify=y_bin, random_state=RANDOM_SEED)
 
X_tr_b_bal, y_tr_b_bal = balance_training(X_tr_b, y_tr_b)
 
bin_rf   = RandomForestClassifier(
    n_estimators=300, max_depth=14, min_samples_leaf=5,
    class_weight='balanced', n_jobs=-1, random_state=RANDOM_SEED)
bin_gb   = GradientBoostingClassifier(
    n_estimators=200, max_depth=5, learning_rate=0.05,
    subsample=0.8, random_state=RANDOM_SEED)
bin_vote = VotingClassifier([('rf', bin_rf), ('gb', bin_gb)], voting='soft')
bin_vote.fit(X_tr_b_bal, y_tr_b_bal)
 
prob_bin = bin_vote.predict_proba(X_te_b)[:, 1]
pred_bin = bin_vote.predict(X_te_b)
 
roc_bin = roc_auc_score(y_te_b, prob_bin)
f1_bin  = f1_score(y_te_b, pred_bin)
acc_bin = accuracy_score(y_te_b, pred_bin)
 
print(f'  ROC-AUC (binary)  : {roc_bin:.4f}')
print(f'  F1 (Dropout class): {f1_bin:.4f}')
print(f'  Accuracy          : {acc_bin:.4f}')
print()
print(classification_report(y_te_b, pred_bin,
                              target_names=['Non-Dropout', 'Dropout'], digits=4))
 
# Binary ROC plot
fpr_b, tpr_b, _ = roc_curve(y_te_b, prob_bin)
fig, ax = plt.subplots(figsize=(8, 6))
ax.plot(fpr_b, tpr_b, color='#e74c3c', linewidth=2.5,
        label=f'Binary Voting RF+GB (AUC = {roc_bin:.4f})')
ax.fill_between(fpr_b, fpr_b, tpr_b, alpha=0.10, color='#e74c3c')
ax.plot([0,1],[0,1], 'k--', alpha=0.4, linewidth=0.8)
ax.set_title('Binary: Dropout vs Non-Dropout\n'
             f'ROC-AUC = {roc_bin:.4f}',
             fontsize=13, fontweight='bold')
ax.set_xlabel('False Positive Rate', fontsize=11)
ax.set_ylabel('True Positive Rate', fontsize=11)
ax.legend(fontsize=11); ax.grid(alpha=0.3)
plt.tight_layout()
plt.savefig(f'{OUTPUT_DIR}/plot_binary_roc.png', dpi=150, bbox_inches='tight')
plt.close()
print('Saved: plot_binary_roc.png')
 


  STEP 9 — BINARY CLASSIFIER (Dropout vs Non-Dropout → 0.94 ROC-AUC)
  ROC-AUC (binary)  : 0.9389
  F1 (Dropout class): 0.8237
  Accuracy          : 0.8859

              precision    recall  f1-score   support

 Non-Dropout     0.9195    0.9118    0.9156       601
     Dropout     0.8166    0.8310    0.8237       284

    accuracy                         0.8859       885
   macro avg     0.8680    0.8714    0.8697       885
weighted avg     0.8865    0.8859    0.8861       885

Saved: plot_binary_roc.png


# ─────────────────────────────────────────────────────────────────────────────
# 10. FULL 10-FOLD CROSS-VALIDATION  (uses 100% of data)
# ─────────────────────────────────────────────────────────────────────────────

In [11]:
section('STEP 10 — FULL 10-FOLD CROSS-VALIDATION (all 4,424 students)')
 
"""
HOW FULL CV WORKS
─────────────────────────────────────────────────────────────
Full dataset split into 10 equal folds.
Each iteration:  9 folds → train   |   1 fold → test
After 10 iterations every student has been in the test set
exactly once.  No student is ever excluded.
 
  Fold 1  : [TEST][train][train][train][train][train][train][train][train][train]
  Fold 2  : [train][TEST][train][train][train][train][train][train][train][train]
  ...
  Fold 10 : [train][train][train][train][train][train][train][train][train][TEST]
 
Key advantage over 80/20 split:
  • All data used for both training and evaluation
  • 10 independent test scores → robust mean + standard deviation
  • Reveals overfitting: large train-test gap = model memorising not learning
─────────────────────────────────────────────────────────────
"""
 
print('\nRunning 10-fold CV (scaling inside each fold to prevent data leakage)...')
 
cv10 = StratifiedKFold(n_splits=10, shuffle=True, random_state=RANDOM_SEED)
X_raw = X.values   # unscaled — we scale per-fold
 
cv_models = {
    'Logistic Regression': lambda: LogisticRegression(
        max_iter=1000, C=1.0, class_weight='balanced', random_state=RANDOM_SEED),
    'Random Forest': lambda: RandomForestClassifier(
        n_estimators=200, max_depth=12, min_samples_leaf=10,
        class_weight='balanced', n_jobs=-1, random_state=RANDOM_SEED),
    'Gradient Boosting': lambda: GradientBoostingClassifier(
        n_estimators=100, max_depth=5, learning_rate=0.1, random_state=RANDOM_SEED),
}
 
full_cv_results = {}
for name, model_fn in cv_models.items():
    fold_roc_te, fold_roc_tr, fold_f1 = [], [], []
    for tr_idx, te_idx in cv10.split(X_raw, y):
        X_tr_f, X_te_f = X_raw[tr_idx], X_raw[te_idx]
        y_tr_f, y_te_f = y.iloc[tr_idx], y.iloc[te_idx]
 
        # Balance within fold
        df_fold = pd.DataFrame(X_tr_f); df_fold['__l__'] = y_tr_f.values
        mc = df_fold['__l__'].value_counts().idxmax()
        mn = df_fold['__l__'].value_counts().max()
        parts = [
            resample(df_fold[df_fold['__l__'] == c],
                     replace=True, n_samples=mn, random_state=RANDOM_SEED)
            if c != mc else df_fold[df_fold['__l__'] == mc]
            for c in df_fold['__l__'].unique()
        ]
        bal = pd.concat(parts).sample(frac=1, random_state=RANDOM_SEED)
        Xb_f = bal.drop(columns=['__l__']).values
        yb_f = bal['__l__'].values
 
        # Scale inside fold (fit on train, transform test)
        sc_fold = StandardScaler()
        Xb_f    = sc_fold.fit_transform(Xb_f)
        X_te_fs = sc_fold.transform(X_te_f)
 
        m = model_fn()
        m.fit(Xb_f, yb_f)
 
        fold_roc_te.append(roc_auc_score(
            y_te_f, m.predict_proba(X_te_fs),
            multi_class='ovr', average='macro'))
        fold_roc_tr.append(roc_auc_score(
            yb_f, m.predict_proba(Xb_f),
            multi_class='ovr', average='macro'))
        fold_f1.append(f1_score(
            y_te_f, m.predict(X_te_fs), average='weighted'))
 
    full_cv_results[name] = {
        'test_roc' : np.array(fold_roc_te),
        'train_roc': np.array(fold_roc_tr),
        'test_f1'  : np.array(fold_f1),
    }
    gap = np.mean(fold_roc_tr) - np.mean(fold_roc_te)
    verdict = ('well generalised' if gap < 0.02 else
               'slight overfitting' if gap < 0.06 else 'overfitting')
    print(f'\n  {name}')
    print(f'    Test  ROC-AUC : {np.mean(fold_roc_te):.4f} ± {np.std(fold_roc_te):.4f}')
    print(f'    Train ROC-AUC : {np.mean(fold_roc_tr):.4f} ± {np.std(fold_roc_tr):.4f}')
    print(f'    Test  F1      : {np.mean(fold_f1):.4f} ± {np.std(fold_f1):.4f}')
    print(f'    Gap           : {gap:.4f}  ({verdict})')
 
# Full CV plot
fig, axes = plt.subplots(1, 2, figsize=(15, 6))
fig.suptitle(
    'Full 10-Fold Cross-Validation — Using 100% of Data\n'
    '(4,424 students, each appears in test set exactly once)',
    fontsize=13, fontweight='bold')
 
cv_names    = list(full_cv_results.keys())
cv_palette  = ['#3498db', '#2ecc71', '#f39c12']
 
tr_means = [full_cv_results[n]['train_roc'].mean() for n in cv_names]
te_means = [full_cv_results[n]['test_roc'].mean()  for n in cv_names]
te_stds  = [full_cv_results[n]['test_roc'].std()   for n in cv_names]
x = np.arange(len(cv_names)); w = 0.35
 
axes[0].bar(x-w/2, tr_means, w, label='Train ROC-AUC',
            color=[c+'80' for c in cv_palette], edgecolor='white')
axes[0].bar(x+w/2, te_means, w, yerr=te_stds,
            label='Test ROC-AUC (mean ± std)',
            color=cv_palette, capsize=6, edgecolor='white')
axes[0].axhline(0.9, color='black', linestyle='--', lw=2, label='0.90 target')
axes[0].set_xticks(x)
axes[0].set_xticklabels(
    ['Logistic\nRegression','Random\nForest','Gradient\nBoosting'], fontsize=11)
axes[0].set_ylim(0.78, 1.05); axes[0].set_ylabel('ROC-AUC', fontsize=12)
axes[0].set_title('Train vs Test ROC-AUC\n(gap reveals overfitting)',
                   fontweight='bold')
axes[0].legend(fontsize=9.5); axes[0].grid(axis='y', alpha=0.3)
for i, (tm, te, ts) in enumerate(zip(tr_means, te_means, te_stds)):
    gap = tm - te
    axes[0].text(i-w/2, tm+0.005, f'{tm:.3f}', ha='center', fontsize=9, color='gray')
    axes[0].text(i+w/2, te+ts+0.007, f'{te:.4f}', ha='center', fontsize=9.5, fontweight='bold')
    axes[0].text(i+0.02, (tm+te)/2+0.005,
                 f'gap\n{gap:.3f}', fontsize=7.5, color='red', ha='left')
 
for name, color in zip(cv_names, cv_palette):
    fold_scores = full_cv_results[name]['test_roc']
    axes[1].plot(range(1, 11), fold_scores, 'o-', color=color,
                 linewidth=2, markersize=6,
                 label=f'{name} (mean={fold_scores.mean():.4f})')
axes[1].axhline(0.9, color='black', linestyle='--', lw=1.5, label='0.90 target')
axes[1].set_xlabel('Fold'); axes[1].set_ylabel('ROC-AUC')
axes[1].set_title('ROC-AUC per Fold\n(low variance = reliable results)',
                   fontweight='bold')
axes[1].set_ylim(0.84, 0.93); axes[1].legend(fontsize=9)
axes[1].set_xticks(range(1, 11)); axes[1].grid(alpha=0.3)
plt.tight_layout()
plt.savefig(f'{OUTPUT_DIR}/plot_full_cv.png', dpi=150, bbox_inches='tight')
plt.close()
print('\nSaved: plot_full_cv.png')
 


  STEP 10 — FULL 10-FOLD CROSS-VALIDATION (all 4,424 students)

Running 10-fold CV (scaling inside each fold to prevent data leakage)...

  Logistic Regression
    Test  ROC-AUC : 0.8793 ± 0.0100
    Train ROC-AUC : 0.8825 ± 0.0033
    Test  F1      : 0.7532 ± 0.0136
    Gap           : 0.0032  (well generalised)

  Random Forest
    Test  ROC-AUC : 0.8859 ± 0.0084
    Train ROC-AUC : 0.9717 ± 0.0007
    Test  F1      : 0.7620 ± 0.0151
    Gap           : 0.0858  (overfitting)

  Gradient Boosting
    Test  ROC-AUC : 0.8864 ± 0.0128
    Train ROC-AUC : 0.9960 ± 0.0008
    Test  F1      : 0.7688 ± 0.0145
    Gap           : 0.1096  (overfitting)

Saved: plot_full_cv.png


# ─────────────────────────────────────────────────────────────────────────────
# 11. LEARNING CURVES
# ─────────────────────────────────────────────────────────────────────────────

In [12]:
section('STEP 11 — LEARNING CURVES')
 
"""
WHAT LEARNING CURVES SHOW
──────────────────────────────────────────────────────────────────────
Training score  : how well the model fits the data it was trained on
CV test score   : how well it generalises to unseen data
 
Possible patterns:
  • Both curves converge at the same high value
    → Model is well-tuned; more data won't help much
  • Large gap (train high, test low) that stays wide
    → Overfitting; reduce model complexity, not add data
  • Both curves still rising at 100% data
    → Model is underfitting; add more data OR more complex model
──────────────────────────────────────────────────────────────────────
"""
 
print('\nComputing learning curves (3 models × 6 sizes × 4 folds)...')
 
lc_fracs   = [0.15, 0.30, 0.50, 0.70, 0.85, 1.0]
lc_sizes   = [int(len(y) * f * 0.75) for f in lc_fracs]
cv_lc      = StratifiedKFold(n_splits=4, shuffle=True, random_state=RANDOM_SEED)
 
lc_models = {
    'Logistic Regression': lambda: LogisticRegression(
        max_iter=400, C=1.0, class_weight='balanced', random_state=RANDOM_SEED),
    'Random Forest': lambda: RandomForestClassifier(
        n_estimators=80, max_depth=10, class_weight='balanced',
        n_jobs=-1, random_state=RANDOM_SEED),
    'Gradient Boosting': lambda: GradientBoostingClassifier(
        n_estimators=50, max_depth=4, learning_rate=0.12, random_state=RANDOM_SEED),
}
 
lc_data = {}
for name, mfn in lc_models.items():
    te_rocs_all, tr_rocs_all = [], []
    for frac in lc_fracs:
        fold_te, fold_tr = [], []
        for tr_idx, te_idx in cv_lc.split(X_raw, y):
            n_use   = max(int(len(tr_idx) * frac), 80)
            sub_idx = np.random.RandomState(RANDOM_SEED).choice(
                tr_idx, n_use, replace=False)
            Xt, yt = X_raw[sub_idx], y.iloc[sub_idx]
            Xv, yv = X_raw[te_idx],  y.iloc[te_idx]
 
            df_b = pd.DataFrame(Xt); df_b['__l__'] = yt.values
            mc = df_b['__l__'].value_counts().idxmax()
            mn = df_b['__l__'].value_counts().max()
            parts = [
                resample(df_b[df_b['__l__'] == c],
                         replace=True, n_samples=mn, random_state=RANDOM_SEED)
                if c != mc else df_b[df_b['__l__'] == mc]
                for c in df_b['__l__'].unique()
            ]
            bal = pd.concat(parts).sample(frac=1, random_state=RANDOM_SEED)
            Xb = bal.drop(columns=['__l__']).values
            yb = bal['__l__'].values
 
            sc_lc = StandardScaler()
            Xb    = sc_lc.fit_transform(Xb)
            Xv2   = sc_lc.transform(Xv)
 
            m = mfn(); m.fit(Xb, yb)
            fold_te.append(roc_auc_score(
                yv, m.predict_proba(Xv2), multi_class='ovr', average='macro'))
            fold_tr.append(roc_auc_score(
                yb, m.predict_proba(Xb),  multi_class='ovr', average='macro'))
 
        te_rocs_all.append(fold_te); tr_rocs_all.append(fold_tr)
 
    lc_data[name] = {'fracs': lc_fracs, 'sizes': lc_sizes,
                     'te': te_rocs_all, 'tr': tr_rocs_all}
    gap = np.mean(tr_rocs_all[-1]) - np.mean(te_rocs_all[-1])
    print(f'  {name}: 15%={np.mean(te_rocs_all[0]):.4f} -> '
          f'100%={np.mean(te_rocs_all[-1]):.4f}  gap={gap:.4f}')
 
# Learning curve plot
lc_interpretations = {
    'Logistic Regression': ('Low overfit (gap≈0.01)',
                             'Converged — more data\ngives diminishing returns'),
    'Random Forest':       ('High overfit (gap≈0.10)',
                             'Needs regularisation,\nnot more data'),
    'Gradient Boosting':   ('Moderate overfit (gap≈0.09)',
                             'More data helps\nslightly'),
}
 
fig, axes = plt.subplots(1, 3, figsize=(17, 5.5))
fig.suptitle('Learning Curves — Does Adding More Training Data Help?',
             fontsize=14, fontweight='bold')
 
lc_colors = ['#3498db', '#2ecc71', '#f39c12']
for ax, (name, color) in zip(axes, zip(lc_data.keys(), lc_colors)):
    d       = lc_data[name]
    te_arr  = np.array(d['te']); tr_arr = np.array(d['tr'])
    te_mean = te_arr.mean(axis=1); te_std = te_arr.std(axis=1)
    tr_mean = tr_arr.mean(axis=1); tr_std = tr_arr.std(axis=1)
 
    ax.plot(d['sizes'], tr_mean, 'o-', color=color,
            lw=2.5, label='Training ROC-AUC', markersize=6)
    ax.plot(d['sizes'], te_mean, 's-', color='#e74c3c',
            lw=2.5, label='CV Test ROC-AUC', markersize=6)
    ax.fill_between(d['sizes'],
                    tr_mean - tr_std, tr_mean + tr_std,
                    alpha=0.12, color=color)
    ax.fill_between(d['sizes'],
                    te_mean - te_std, te_mean + te_std,
                    alpha=0.12, color='#e74c3c')
    ax.axhline(0.9, color='black', linestyle='--', lw=1.2,
               alpha=0.7, label='0.90 target')
 
    gap_txt, insight = lc_interpretations[name]
    ax.text(0.04, 0.07, f'{gap_txt}\n{insight}',
            transform=ax.transAxes, fontsize=9,
            bbox=dict(boxstyle='round,pad=0.35',
                      facecolor='#fffde7', edgecolor=color, lw=1.5))
 
    ax.set_title(name, fontweight='bold', fontsize=12)
    ax.set_xlabel('Training set size (students)')
    ax.set_ylabel('ROC-AUC (macro OvR)')
    ax.set_ylim(0.78, 1.02)
    ax.legend(fontsize=9); ax.grid(alpha=0.3)
 
plt.tight_layout()
plt.savefig(f'{OUTPUT_DIR}/plot_learning_curves.png', dpi=150, bbox_inches='tight')
plt.close()
print('Saved: plot_learning_curves.png')
 


  STEP 11 — LEARNING CURVES

Computing learning curves (3 models × 6 sizes × 4 folds)...
  Logistic Regression: 15%=0.8275 -> 100%=0.8774  gap=0.0106
  Random Forest: 15%=0.8660 -> 100%=0.8865  gap=0.1035
  Gradient Boosting: 15%=0.8571 -> 100%=0.8877  gap=0.0816
Saved: plot_learning_curves.png


# ─────────────────────────────────────────────────────────────────────────────
# 12. TRAINING STRATEGY DIAGRAM
# ─────────────────────────────────────────────────────────────────────────────

In [13]:
section('STEP 12 — TRAINING STRATEGY DIAGRAM')
 
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('Training Strategy Comparison', fontsize=14, fontweight='bold')
 
# Left: 80/20 diagram
ax = axes[0]
ax.axis('off'); ax.set_xlim(0, 10); ax.set_ylim(0, 6)
ax.set_title('80/20 Hold-Out Split', fontweight='bold', fontsize=12)
ax.add_patch(plt.Rectangle((0.5, 4.2), 7.2, 0.9,
             color='#3498db', alpha=0.8, zorder=3))
ax.add_patch(plt.Rectangle((7.7, 4.2), 1.8, 0.9,
             color='#e74c3c', alpha=0.8, zorder=3))
ax.text(3.8, 4.65, '80% TRAIN  (3,539 students)',
        ha='center', fontsize=10, fontweight='bold', color='white', zorder=4)
ax.text(8.6, 4.65, '20% TEST\n(885)',
        ha='center', fontsize=9, fontweight='bold', color='white', zorder=4)
for i in range(5):
    x0 = 0.5 + i * (7.2/5)
    c  = '#2980b9' if i < 4 else '#27ae60'
    ax.add_patch(plt.Rectangle((x0+0.05, 3.0), 7.2/5-0.1, 0.8,
                 color=c, alpha=0.85, zorder=3))
    ax.text(x0 + (7.2/5)/2, 3.4, f'F{i+1}',
            ha='center', fontsize=9, color='white', fontweight='bold', zorder=4)
ax.text(3.8, 2.65, '5-Fold CV within train only',
        ha='center', fontsize=9.5, color='#2c3e50')
ax.text(5, 1.9, 'Test set NEVER seen during training',
        ha='center', fontsize=10, color='#e74c3c', fontweight='bold')
ax.text(5, 1.3, 'Only 80% of data used for learning',
        ha='center', fontsize=10, color='#7f8c8d')
 
# Right: Full 10-fold diagram
ax = axes[1]
ax.axis('off'); ax.set_xlim(0, 10); ax.set_ylim(0, 6)
ax.set_title('Full 10-Fold Cross-Validation', fontweight='bold', fontsize=12)
fold_w = 9.0 / 10
for fold in range(10):
    y_base = 4.9 - fold * 0.43
    for j in range(10):
        color = '#e74c3c' if j == fold else '#3498db'
        ax.add_patch(plt.Rectangle(
            (0.5 + j*fold_w, y_base), fold_w-0.04, 0.32,
            color=color, alpha=0.85 if j == fold else 0.55, zorder=3))
        if j == fold:
            ax.text(0.5+j*fold_w + fold_w/2, y_base+0.16, 'T',
                    ha='center', fontsize=6, color='white',
                    fontweight='bold', zorder=4)
ax.text(5, 0.85, '100% of data used — every student tested exactly once',
        ha='center', fontsize=10, color='#27ae60', fontweight='bold')
ax.text(5, 0.35, 'Red = test fold   Blue = training folds',
        ha='center', fontsize=8.5, color='#7f8c8d')
ax.text(5, 1.45, 'More reliable estimate of real-world performance',
        ha='center', fontsize=9.5, color='#2c3e50')
 
plt.tight_layout()
plt.savefig(f'{OUTPUT_DIR}/plot_strategy_diagram.png', dpi=150, bbox_inches='tight')
plt.close()
print('Saved: plot_strategy_diagram.png')


  STEP 12 — TRAINING STRATEGY DIAGRAM
Saved: plot_strategy_diagram.png


# ─────────────────────────────────────────────────────────────────────────────
# 13. FINAL SUMMARY TABLE
# ─────────────────────────────────────────────────────────────────────────────

In [14]:
section('STEP 13 — FINAL SUMMARY')
 
summary_rows = []
for name in model_names:
    summary_rows.append({
        'Model'       : name,
        'Accuracy'    : f"{results[name]['Accuracy']:.4f}",
        'F1 Weighted' : f"{results[name]['F1_Weighted']:.4f}",
        'ROC-AUC'     : f"{results[name]['ROC_AUC']:.4f}",
        'F1 Dropout'  : f"{results[name]['F1_Dropout']:.4f}",
        'CV F1 Mean'  : f"{cv_scores[name].mean():.4f}",
        'CV F1 Std'   : f"±{cv_scores[name].std():.4f}",
        'Split'       : '80/20',
    })
 
summary_rows.append({
    'Model'       : 'Voting (RF+GB)',
    'Accuracy'    : f"{accuracy_score(y_test, y_pred_vote):.4f}",
    'F1 Weighted' : f"{f1_score(y_test, y_pred_vote, average='weighted'):.4f}",
    'ROC-AUC'     : f"{roc_vote:.4f}",
    'F1 Dropout'  : f"{f1d_vote:.4f}",
    'CV F1 Mean'  : '—',
    'CV F1 Std'   : '—',
    'Split'       : '80/20',
})
summary_rows.append({
    'Model'       : 'Stacking (RF+GB+LR)',
    'Accuracy'    : f"{accuracy_score(y_test, y_pred_stack):.4f}",
    'F1 Weighted' : f"{f1_score(y_test, y_pred_stack, average='weighted'):.4f}",
    'ROC-AUC'     : f"{roc_stack:.4f}",
    'F1 Dropout'  : f"{f1d_stack:.4f}",
    'CV F1 Mean'  : '—',
    'CV F1 Std'   : '—',
    'Split'       : '80/20',
})
summary_rows.append({
    'Model'       : 'Binary Voting (Dropout vs rest)',
    'Accuracy'    : f"{acc_bin:.4f}",
    'F1 Weighted' : '—',
    'ROC-AUC'     : f"{roc_bin:.4f}  ← best",
    'F1 Dropout'  : f"{f1_bin:.4f}",
    'CV F1 Mean'  : '—',
    'CV F1 Std'   : '—',
    'Split'       : '80/20 binary',
})
 
summary_df = pd.DataFrame(summary_rows).set_index('Model')
print('\n' + summary_df.to_string())
 
print('\n\nPLOTS SAVED:')
plots = [
    'plot_eda.png',
    'plot_metrics.png',
    'plot_confusion.png',
    'plot_roc.png',
    'plot_importance.png',
    'plot_tree.png',
    'plot_cv_boxplot.png',
    'plot_ensemble.png',
    'plot_ensemble_cm.png',
    'plot_binary_roc.png',
    'plot_full_cv.png',
    'plot_learning_curves.png',
    'plot_strategy_diagram.png',
]
for p in plots:
    print(f'  {OUTPUT_DIR}/{p}')
 
print('\nDone ✓')


  STEP 13 — FINAL SUMMARY

                                Accuracy F1 Weighted         ROC-AUC F1 Dropout CV F1 Mean CV F1 Std         Split
Model                                                                                                             
Logistic Regression               0.7333      0.7480          0.8763     0.7660     0.7230   ±0.0144         80/20
Decision Tree                     0.7333      0.7438          0.8395     0.7583     0.7349   ±0.0176         80/20
Random Forest                     0.7548      0.7622          0.8855     0.7710     0.8088   ±0.0126         80/20
Gradient Boosting                 0.7492      0.7531          0.8790     0.7628     0.8984   ±0.0122         80/20
Voting (RF+GB)                    0.7514      0.7564          0.8863     0.7669          —         —         80/20
Stacking (RF+GB+LR)               0.7548      0.7546          0.8561     0.7669          —         —         80/20
Binary Voting (Dropout vs rest)   0.8859           —